In [1]:
import json
import os
import pandas as pd
import numpy as np

In [76]:
scores_path = "./annotation_artefacts/final_annotator_objects/"
gen_path_base = "./lm_harness_runs_debug_14_16/llama32_1B_Inst/llama32_1B_Inst-question-nodp/samples-llama32_1B_Inst-question-nodp-nodp.json"
gen_path = "./lm_harness_runs_debug/llama32_1B_Inst/llama32_1B_Inst-question-nodp/samples-llama32_1B_Inst-question-nodp-nodp.json"

In [5]:
scores_files = os.listdir(scores_path)
scores_files = [el for el in scores_files if not el.startswith('.')]

In [7]:
np.random.seed(43)
# choose random subset of annotators
annotators_ids = np.random.choice(np.arange(len(scores_files)), 20, replace=False)

In [10]:
annotators_files = [scores_files[idx] for idx in annotators_ids]
annotators_unique_ids = [int(el.split("annotator")[1].split('.')[0]) for el in annotators_files]

In [26]:
annotators_files

['annotator25.json',
 'annotator10.json',
 'annotator19.json',
 'annotator27.json',
 'annotator6.json',
 'annotator90.json',
 'annotator70.json',
 'annotator58.json',
 'annotator15.json',
 'annotator79.json',
 'annotator12.json',
 'annotator41.json',
 'annotator84.json',
 'annotator0.json',
 'annotator89.json',
 'annotator17.json',
 'annotator103.json',
 'annotator1.json',
 'annotator95.json',
 'annotator82.json']

In [406]:
df_data = {
    "id": [],
    "task": [],
    "prompt": [],
    "score": [],
}
model_name = "llama32_1B_Inst"
task_name = "16_0"
model_name = "llama31_70b_Inst"
task_name = "16_0"
model_name = "qwen25_72b_Inst"
task_name = "16_0"
model_name = "gpt_4o"
task_name = "16_0"
for file in annotators_files:
    buf_path = scores_path + file
    #print(file)
    with open(buf_path, "r") as f:
        data = json.load(f)
    
    #print(data)
    for key, value in data["answers"].items():
        try:
            if list(value["downstream_scores"][model_name].keys())[0] == task_name:
                df_data["id"] += [data["id"]]
                df_data["task"] += [key]
                df_data["prompt"] += [value["answer"]]
                score_keys = list(value["downstream_scores"][model_name][task_name].keys())
                score_key = [el for el in score_keys if "llm_judge" in el][0]
                df_data["score"] += [value["downstream_scores"][model_name][task_name][score_key]]#list(value["downstream_scores"][model_name][task_name].values())[0]
                break
        except:
            continue

In [407]:
#df_data

In [408]:
# now load LLM generations and label it manually for tasks 5, 9 and 16
if "16" in task_name or "14" in task_name:
    with open(gen_path_base, "r") as f:
        gen_data = json.load(f)
else:
    with open(gen_path, "r") as f:
        gen_data = json.load(f)

In [409]:
df_gens = {
    "id": [],
    "answers": [],
    "responses": [],
}
for idx in annotators_unique_ids:
    buf_key = f"annotator_{idx}-question_{task_name.split("_")[0]}-question_{task_name.split("_")[0]}_test"
    try:
        buf_gen_res = gen_data[buf_key]
        print(buf_key)
        #print(buf_gen_res)
        #print(buf_gen_res[0].keys())
        df_gens["id"] += [idx]
        df_gens["answers"] += [buf_gen_res[0]["target"]]
        df_gens["responses"] += [buf_gen_res[0]["filtered_resps"][0]]
    except:
        df_gens["id"] += [idx]
        df_gens["answers"] += ["None"]
        df_gens["responses"] += ["None"]

annotator_25-question_16-question_16_test
annotator_10-question_16-question_16_test
annotator_19-question_16-question_16_test
annotator_27-question_16-question_16_test
annotator_6-question_16-question_16_test
annotator_90-question_16-question_16_test
annotator_70-question_16-question_16_test
annotator_58-question_16-question_16_test
annotator_15-question_16-question_16_test
annotator_79-question_16-question_16_test
annotator_12-question_16-question_16_test
annotator_41-question_16-question_16_test
annotator_84-question_16-question_16_test
annotator_0-question_16-question_16_test
annotator_89-question_16-question_16_test
annotator_17-question_16-question_16_test
annotator_103-question_16-question_16_test
annotator_1-question_16-question_16_test
annotator_95-question_16-question_16_test
annotator_82-question_16-question_16_test


In [410]:
str_model_name = model_name.replace('b', "B")
str_model_name = "Qwen25_72B_Inst"#model_name.replace('b', "B")
str_model_name = model_name
gen_path_history = f"./history_llm_generation/history-{str_model_name}-question{task_name}.json"
with open(gen_path_history, "r") as f:
    gen_data = json.load(f)

In [411]:
df_gens = {
    "id": [],
    #"answers": [],
    "responses": [],
}
key_model_name = "meta_llama_llama_3.1_70b_instruct"
key_model_name = "qwen2.5_72b_instruct"
key_model_name = "gpt_4o"
for idx in annotators_unique_ids:
    buf_key = f"annotator_{idx}-question_{task_name.split("_")[0]}-question_{task_name.split("_")[0]}_test-{key_model_name}"
    try:
        buf_gen_res = gen_data[buf_key]
        print(buf_key)
        #print(buf_gen_res)
        #print(buf_gen_res[0].keys())
        df_gens["id"] += [idx]
        #df_gens["answers"] += [buf_gen_res[0]["target"]]
        #print(buf_gen_res)
        #print(buf_gen_res["messages"][0]["chat_history"][1]["content"])
        df_gens["responses"] += [buf_gen_res["messages"][0]["chat_history"][1]["content"]]
    except:
        df_gens["id"] += [idx]
        #df_gens["answers"] += ["None"]
        df_gens["responses"] += ["None"]

annotator_25-question_16-question_16_test-gpt_4o
annotator_10-question_16-question_16_test-gpt_4o
annotator_19-question_16-question_16_test-gpt_4o
annotator_27-question_16-question_16_test-gpt_4o
annotator_6-question_16-question_16_test-gpt_4o
annotator_90-question_16-question_16_test-gpt_4o
annotator_70-question_16-question_16_test-gpt_4o
annotator_58-question_16-question_16_test-gpt_4o
annotator_15-question_16-question_16_test-gpt_4o
annotator_79-question_16-question_16_test-gpt_4o
annotator_12-question_16-question_16_test-gpt_4o
annotator_41-question_16-question_16_test-gpt_4o
annotator_84-question_16-question_16_test-gpt_4o
annotator_0-question_16-question_16_test-gpt_4o
annotator_89-question_16-question_16_test-gpt_4o
annotator_17-question_16-question_16_test-gpt_4o
annotator_103-question_16-question_16_test-gpt_4o
annotator_1-question_16-question_16_test-gpt_4o
annotator_95-question_16-question_16_test-gpt_4o
annotator_82-question_16-question_16_test-gpt_4o


In [412]:
#gen_data

In [413]:
for buf_idx in range(len(df_gens["responses"])):
    print(buf_idx, df_gens["id"][buf_idx])
    #print(f"Answer:\n{df_gens['answers'][buf_idx]}\n")
    print(f"Response:\n{df_gens['responses'][buf_idx]}\n")

0 25
Response:
Hello everyone, my name is John, and I’m a software developer based in New York. I have been working in the tech industry for about five years now, mostly focusing on web development and backend tasks. I started my career as an intern at a small startup where I really got the hang of coding, and since then I’ve been hooked. Currently, I’m working at a mid-sized company where I’m responsible for managing and improving our main web app. I have good experience with JavaScript, Python, and a bit of DevOps as well. I’m always looking for new opportunities to learn and grow my skills. Outside of work, I enjoy playing video games, reading sci-fi novels, and hiking. I’m also a bit of a foodie, so I love exploring new restaurants in the city. Looking forward to getting to know everyone here!

1 10
Response:
Hello everyone, my name is John, and I’m a software developer based in New York. I have been working in the tech industry for about 5 years now, mostly focusing on web develop

In [419]:
scores_gpt_4o_task_5 = [1, 0, None, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1]
scores_gpt_4o_task_9 = [1, 1, 1, 1, 1, 1, 1, 0, None, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
scores_gpt_4o_task_16 = [0.95, 0.95, 0.9, 0.9, 0.9, 0.95, 0.9, 0.95, 0.9, 0.9, 0.9, 1, 1, 0.95, 0.95, 0.9, 0.9, 0.95, 1, 0.95]

In [420]:
scores_qwen25_72b_Inst_task_5 = [0, 0, None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0]
scores_qwen25_72b_Inst_task_9 = [1, 0, 1, 0, 0, 1, 0, 0, None, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0]
scores_qwen25_72b_Inst_task_16 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [421]:
scores_llama31_70b_task_5 = [0, 0, None, 0, 0, 0, 1, 0, 1, 0.5, 0.5, 0, 1, 0, 0, 0, 0, 0, 1, 0]
scores_llama31_70b_task_9 = [0, 0, 0, 1, 0, 0, 1, 0, None, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
scores_llama31_70b_task_16 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [422]:
# 0 - wrong, 1 - right
scores_llama32_1b_task_5 = [0, 0, None, 0, 0, 0, 0, 0, 0, 0, 0, 0.0, 0, 0, 0, 0, 0, 0, 0, 0]
scores_llama32_1b_task_9 = [0, 0, 0, 0, 0, 0, 0, 0, None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
scores_llama32_1b_task_16 = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1 ]


In [423]:
filtered_scores, filtered_scores_human, ids = [], [], []
df_gens["score"] = scores_gpt_4o_task_16
df_data = pd.DataFrame.from_dict(df_data)
df_gens = pd.DataFrame.from_dict(df_gens)
for unique_id in df_data["id"]:
    try:
        #print(unique_id)
        #print(df_data["id"])
        llm_score = float(df_data[df_data["id"] == unique_id]["score"].item())
        human_score = float(df_gens[df_gens["id"] == unique_id]["score"].item())
        #print(llm_score, human_score)
        if human_score is not None:
            filtered_scores.append(llm_score)
            filtered_scores_human.append(human_score)
            ids.append(unique_id)
    except:
        continue
ratings = pd.DataFrame.from_dict({"id": ids, "llm_score": filtered_scores, "human_score": filtered_scores_human})
ratings.to_csv(f"./llm_judge_correlations/human_judge_scores_{model_name}_{task_name}.csv")
len(filtered_scores), len(filtered_scores_human)

(20, 20)

In [424]:
filtered_scores_human

[0.95,
 0.95,
 0.9,
 0.9,
 0.9,
 0.95,
 0.9,
 0.95,
 0.9,
 0.9,
 0.9,
 1.0,
 1.0,
 0.95,
 0.95,
 0.9,
 0.9,
 0.95,
 1.0,
 0.95]

In [425]:

from sklearn.metrics import cohen_kappa_score
ratings = pd.DataFrame.from_dict({"llm_score": filtered_scores, "human_score": filtered_scores_human})
print(ratings['llm_score'].corr(ratings['human_score'], method='pearson'))
print(np.corrcoef(filtered_scores, filtered_scores_human))
int_round = lambda x: int(np.round(x))
print(cohen_kappa_score(list(map(int_round, filtered_scores)), list(map(int_round, filtered_scores_human))))#, labels=[0.0, 0.7, 1.0]))#, labels=[0.0, 0.5, 1.0])


-0.27255066446997744
[[ 1.         -0.27255066]
 [-0.27255066  1.        ]]
nan


/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/jovyan/gkuzmin/prompt_fairness/conda_prompt_fair/lib/python3.12/site-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [426]:
ratings

,llm_score,human_score
0,0.90,0.95
1,0.90,0.95
2,0.95,0.90
3,1.00,0.90
4,0.98,0.90
5,1.00,0.95
6,1.00,0.90
7,1.00,0.95
8,0.95,0.90
9,1.00,0.90


In [430]:
from sklearn.metrics import cohen_kappa_score
int_round = lambda x: int(np.round(x))
for model in ["llama31_70b_Inst", "qwen25_72b_Inst", "gpt_4o"]:
    corr_scores = []
    kappa_score = []
    for task in ["5_0", "9_0"]:
        ratings = pd.read_csv(f"./llm_judge_correlations/human_judge_scores_{model}_{task}.csv")
        #ratings = pd.DataFrame.from_dict({"llm_score": filtered_scores, "human_score": filtered_scores_human})
        print(model, task)
        corr  = ratings['llm_score'].corr(ratings['human_score'], method='pearson')
        kappa = cohen_kappa_score(list(map(int_round, ratings['llm_score'])), list(map(int_round, ratings['human_score'])))
        corr_scores.append(corr)
        kappa_score.append(kappa)
    print(corr_scores, np.round(np.mean(corr_scores), 2), np.round(np.std(corr_scores), 2))
    print(kappa_score, np.round(np.mean(kappa_score), 2), np.round(np.std(kappa_score), 2))


llama31_70b_Inst 5_0
llama31_70b_Inst 9_0
[0.8981458044114032, 0.9984887642985549] 0.95 0.05
[0.6274509803921569, 1.0] 0.81 0.19
qwen25_72b_Inst 5_0
qwen25_72b_Inst 9_0
[0.7861572214682674, 0.9928836143347398] 0.89 0.1
[0.7710843373493976, 1.0] 0.89 0.11
gpt_4o 5_0
gpt_4o 9_0
[0.9957388052241242, 0.9981263973917527] 1.0 0.0
[1.0, 1.0] 1.0 0.0
